# Õppetund 04 - Tööriistade kasutamise disainimuster

Selles õppetükis õpid Microsoft Agent Frameworki (Python) abil AI agentide **tööriistade kasutamise** disainimustrit. Käsitleme:

- Funktsioonitööriistade määratlemist `@tool` dekoratsiooni ja tüübiga parameetritega
- Tööriistade skeemide pakkumist, et mudel teaks, mida iga tööriist teeb
- Tööriistade täitmise kontrollimist kasutades `approval_mode`
- **Struktureeritud väljundi** tagastamist Pydantici mudelite ja `response_format` abil

Stsenaariumiks on **reisibroneerimise agent**, kes suudab otsida sihtkohti, kontrollida saadavust ja hankida lennuinfot.


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Tööriistade määratlemine kasutades @tool dekoratiivset märgendit

`@tool` dekoratiivne märgend muudab tavalise Python funktsiooni tööriistaks, mida agent saab kutsuda.
Olulised punktid:

- **Docstring** saab tööriista kirjelduseks, mida mudel näeb.
- **Tüübimärgendid** (sh `Annotated` koos kirjeldustega) määravad tööriista skeemi.
- `approval_mode` kontrollib, kas kasutaja peab iga kutse enne selle täitmist heaks kiitma.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Mitme tööriistaga agendi loomine

Edastage kõik kolm tööriista kliendile, nii et mudel saab kutsuda kõiki vajalikke tööriistu kasutaja küsimusele vastamiseks.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Struktureeritud väljund tööriistadega

Määrates `response_format` Pydantic mudeliks, sunnitakse agent tagastama hästi tüübistatud JSON-objekti vabatekstilise vormingu asemel. See on kasulik, kui alamprotsessid vajavad tulemuse programmilist töötlemist.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Tööriista kinnitamise mustrid

Parameeter `approval_mode` märgisel `@tool` kontrollib, kas tööriista üleskutsetel on vaja enne täitmist inimese kinnitust:

| Režiim | Käitumine |
|---|---|
| `"never_require"` | Tööriist töötab automaatselt — kasutaja kinnitust ei nõuta. |
| `"always_require"` | Iga üleskutse peab enne täitmist saama kasutaja kinnituse. |

Kasutage `"always_require"` tööriistade puhul, millel on kõrvalmõjusid (nt lennu broneerimine, krediitkaardi laadimine), et inimene oleks protsessis kaasatud.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Määratleda tööriistu** kasutades `@tool` dekoratsiooni tüübiga parameetrite ja docstringidega, mis toimivad tööriista skeemina.
2. **Koostada mitut tööriista** nii, et agent saab neid järjestikku kutsuda keerukate päringute vastamiseks.
3. **Tagastada struktureeritud väljundit** edastades Pydantic mudeli kui `response_format`.
4. **Kontrollida tööriista heakskiitu** `approval_mode` abil, et hoida tundlike operatsioonide puhul inimeste osalus.

Neid mustreid kasutades saab luua usaldusväärseid, tootmiskõlblikke agente, kes suudavad turvaliselt suhelda välissüsteemidega.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
